# Update Hourly Commodity Supply in `nodes_*.json`

Reads per-region annual supply CSVs (units: **tons/hour**) and writes `supply.seg1.max`
values into the matching node instances in `nodes_1.json` through `nodes_7.json`.

**Commodities covered:** AluminumScrap, SteelScrap

**Stage → year mapping** is loaded from `../stage_year_map.csv`.

In [1]:
import csv
import json
from pathlib import Path

import pandas as pd

## Configuration

In [2]:
CSV_DIR    = Path(".")   # supply CSVs live here
SYSTEM_DIR = Path("..")  # nodes_*.json and stage_year_map.csv live here

# node type  ->  (supply CSV filename,  node-id prefix)
COMMODITY_MAP = {
    "AluminumScrap": ("aluminumscrap_supply_5yeartimesteps.csv", "aluminumscrap_source_"),
    "SteelScrap":    ("steelscrapsupply_5yeartimesteps.csv",     "steelscrap_source_"),
}

## Load stage to year mapping

In [3]:
stage_year_df = pd.read_csv(SYSTEM_DIR / "stage_year_map.csv")
STAGE_YEAR = dict(zip(stage_year_df["stage_index"], stage_year_df["year"]))

print("Stage -> year mapping:")
for stage, year in sorted(STAGE_YEAR.items()):
    print(f"  nodes_{stage}.json  ->  {year}")

Stage -> year mapping:
  nodes_1.json  ->  2030
  nodes_2.json  ->  2035
  nodes_3.json  ->  2040
  nodes_4.json  ->  2045
  nodes_5.json  ->  2050
  nodes_6.json  ->  2055
  nodes_7.json  ->  2060


## Load supply CSVs

Each CSV has the form:
```
Unit: tons/hour, 2025, 2030, ..., 2060
Region1Beijing,  val,  val, ..., val
...
```

In [4]:
def load_supply_csv(filename: str) -> dict[str, dict[int, float]]:
    """Return {region: {year: max_supply}} from a supply CSV."""
    path = CSV_DIR / filename
    supply: dict[str, dict[int, float]] = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)  # first column is label, rest are years
        years = [int(h) for h in header[1:]]
        for row in reader:
            region = row[0]
            supply[region] = {
                yr: float(val.replace(",", "")) if val.strip() else 0.0
                for yr, val in zip(years, row[1:])
            }
    return supply

In [5]:
supply_tables: dict[str, dict[str, dict[int, float]]] = {
    node_type: load_supply_csv(csv_file)
    for node_type, (csv_file, _) in COMMODITY_MAP.items()
}

print("Loaded supply tables:")
for node_type, table in supply_tables.items():
    years = sorted(next(iter(table.values())).keys())
    print(f"  {node_type:14s}  {len(table)} regions  years: {years}")

Loaded supply tables:
  AluminumScrap   31 regions  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]
  SteelScrap      31 regions  years: [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]


## Preview supply data (optional)

Change `commodity_to_preview` to inspect a different commodity.

In [6]:
commodity_to_preview = "AluminumScrap"

df = pd.DataFrame(supply_tables[commodity_to_preview]).T
df.index.name = "Region"
df.columns = [int(c) for c in df.columns]
df.sort_index(inplace=True)
df

,2025,2030,2035,2040,2045,2050,2055,2060
Region,,,,,,,,
Region10Jiangsu,81.99,137.57,208.31,258.43,264.18,266.89,276.96,365.44
Region11Zhejiang,71.18,119.43,180.84,224.35,229.34,231.69,240.44,317.25
Region12Anhui,76.19,127.83,193.56,240.14,245.48,248.00,257.36,339.57
Region13Fujian,52.17,87.53,132.55,164.44,168.09,169.82,176.23,232.52
Region14Jiangxi,53.86,90.37,136.83,169.76,173.53,175.31,181.93,240.04
Region15Shandong,85.75,143.87,217.85,270.27,276.28,279.12,289.65,382.18
Region16Henan,63.19,106.02,160.54,199.17,203.60,205.69,213.45,281.64
Region17Hubei,52.76,88.53,134.05,166.30,170.00,171.74,178.22,235.16
Region18Hunan,59.30,99.49,150.65,186.89,191.05,193.01,200.29,264.28


## Update `nodes_*.json`

Writes the CSV value into `supply.seg1.max[0]` for each matching node instance.

In [ ]:
def update_nodes_file(
    stage: int,
    year: int,
    supply_tables: dict[str, dict[str, dict[int, float]]],
) -> dict[str, int]:
    """Update supply.seg1.max for all supply nodes in nodes_{stage}.json.

    Returns {node_type: count_updated}.
    """
    path = SYSTEM_DIR / f"nodes_{stage}.json"
    with open(path) as f:
        data = json.load(f)

    counts: dict[str, int] = {t: 0 for t in COMMODITY_MAP}

    for node in data["nodes"]:
        node_type = node.get("type")
        if node_type not in COMMODITY_MAP:
            continue

        _, prefix = COMMODITY_MAP[node_type]
        table = supply_tables[node_type]

        for inst in node.get("instance_data", []):
            node_id = inst.get("id", "")
            if not node_id.startswith(prefix):
                continue
            region = node_id[len(prefix):]
            if region not in table or year not in table[region]:
                print(f"  WARNING: no data for {node_type} / {region} / {year}")
                continue

            inst["supply"]["seg1"]["max"][0] = table[region][year]
            counts[node_type] += 1

        csv_order = {region: i for i, region in enumerate(table.keys())}
        node["instance_data"].sort(
            key=lambda x: csv_order.get(x.get("id", "")[len(prefix):], len(csv_order))
        )

    with open(path, "w") as f:
        json.dump(data, f, indent=2)

    return counts

In [8]:
summary_rows = []

for stage, year in sorted(STAGE_YEAR.items()):
    counts = update_nodes_file(stage, year, supply_tables)
    total  = sum(counts.values())
    detail = ", ".join(f"{t}: {n}" for t, n in counts.items() if n)
    print(f"nodes_{stage}.json (year {year}): updated {total} nodes  [{detail}]")
    summary_rows.append({"stage": stage, "year": year, **counts, "total": total})

print("\nDone.")

nodes_1.json (year 2030): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_2.json (year 2035): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_3.json (year 2040): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_4.json (year 2045): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_5.json (year 2050): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_6.json (year 2055): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]
nodes_7.json (year 2060): updated 62 nodes  [AluminumScrap: 31, SteelScrap: 31]

Done.


## Summary

In [9]:
summary_df = pd.DataFrame(summary_rows).set_index(["stage", "year"])
summary_df

,,AluminumScrap,SteelScrap,total
stage,year,,,
1,2030,31,31,62
2,2035,31,31,62
3,2040,31,31,62
4,2045,31,31,62
5,2050,31,31,62
6,2055,31,31,62
7,2060,31,31,62


## Spot-check: verify written values

Reads back the file and confirms the written value matches the source CSV.

In [10]:
check_stage     = 1
check_commodity = "AluminumScrap"
check_region    = "Region1Beijing"

_, prefix = COMMODITY_MAP[check_commodity]
node_id   = prefix + check_region
year      = STAGE_YEAR[check_stage]

with open(SYSTEM_DIR / f"nodes_{check_stage}.json") as f:
    data = json.load(f)

for node in data["nodes"]:
    if node.get("type") == check_commodity:
        for inst in node["instance_data"]:
            if inst["id"] == node_id:
                written  = inst["supply"]["seg1"]["max"][0]
                expected = supply_tables[check_commodity][check_region][year]
                status   = "OK" if written == expected else "MISMATCH"
                print(f"{node_id}  year={year}  written={written}  expected={expected}  [{status}]")

aluminumscrap_source_Region1Beijing  year=2030  written=2.58  expected=2.58  [OK]
